# 00. 환경 점검 — 점심 전에 5분

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/00_env_check.ipynb)

> 🧭 **스토리** — 오후 실습이 매끄럽게 시작되도록 GPU 할당, API 키, 데이터 자동 로드, Ollama 설치를 미리 확인한다. 끝나면 **런타임 연결 해제**까지.
>
> 📖 **이론 연결** — 준비 단계 (이론 없음) · ⏱ **5분** · 🖥 **GPU 필수**
>
> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. (GPU가 없으면 작은 설정으로 느리게 돌아갑니다.)

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 1. GPU 확인 ⚙️
위 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 를 고른 뒤 실행하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'GPU 없음: 런타임 유형을 T4로 바꾸세요'

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

## 2. 실습 데이터 자동 로드 확인 ⚙️
오늘 모든 노트북은 깃허브(`gunyang-data/genai`)에서 데이터를 **자동으로** 받습니다. 업로드·드라이브 마운트가 필요 없습니다.

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
qa = load_json("hanbit_qa.json")
print(f"평가 질문 {len(qa)}개 로드 OK — 예: {qa[0]['q']}")
print("규정집 마크다운:", len(load_text("hanbit_rules_gt.md")), "자")

## 3. OpenAI API 키 확인 ⚙️
수업에서 나눠 준 키를 붙여 넣으세요. 키는 화면에 보이지 않고, 노트북에 저장되지 않습니다.
(`models.retrieve`는 토큰을 쓰지 않는 확인 호출입니다.)

In [ ]:
%pip -q install openai tiktoken

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
from openai import OpenAI
try:
    m = OpenAI().models.retrieve(MODEL_API)
    print("키 OK · 모델 확인:", m.id)
except Exception as e:
    print("키 확인 실패 — 강사에게 알려 주세요:", type(e).__name__, getattr(e, "status_code", ""))  # 오류 본문은 키 일부를 포함할 수 있어 출력하지 않습니다

## 4. Ollama 설치 체감 ⚙️
오후 4·11·13번 노트북에서 쓰는 로컬 LLM 서버입니다. 설치만 해 보고(모델은 오후에 받습니다) 몇 초 걸리는지 봅니다.

In [ ]:
# ⚙️ Ollama 설치 + 서버 실행 (코랩 리눅스 VM 안에서). 첫 실행 1분 안팎.
import subprocess, shutil, time, os, urllib.request
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# 서버를 백그라운드로 띄우고 준비될 때까지 기다립니다
if not os.environ.get("_OLLAMA_STARTED"):
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.environ["_OLLAMA_STARTED"] = "1"
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama 준비 완료:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

## 5. 끝 — 런타임 정리 ✅
무료 코랩은 GPU 세션을 **하나만** 허용합니다. 점심 전에 **런타임 → 런타임 연결 해제 및 삭제** 를 눌러 두세요.
오후 첫 노트북(`02_tokenizer`)은 CPU로 돌아가므로 GPU가 없어도 됩니다.

In [ ]:
print("환경 점검 완료 ✅  이제 '런타임 → 런타임 연결 해제 및 삭제' 를 누르세요.")